# Notebook 05 — Modelisation & Apprentissage

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook implemente les **deux branches de modelisation** du pipeline :

**Branche A — Machine Learning tabulaire**
1. Split stratifie 80/20
2. Baseline majoritaire
3. Regression logistique (Pipeline + StandardScaler)
4. Random Forest Classifier

**Branche B — Deep Learning Vision (CNN)**
5. Generation d'images synthetiques (Cercle vs. Rectangles)
6. Architecture CNN TensorFlow (2 blocs Conv+Pool)
7. Entrainement sur 10 epoques

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, models

from utils_viz import set_custom_style

set_custom_style(theme='light')
%matplotlib inline
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print('Environnement pret. TensorFlow :', tf.__version__)

---

## BRANCHE A — Machine Learning Tabulaire

### A1. Chargement et split stratifie

In [ ]:
df = pd.read_csv('data/processed/tp1_student_risk_model_ready.csv')
y = df['dropout_risk'].astype(int)
X = df.drop(columns=['dropout_risk'])

# Split stratifie : preserve la proportion de positifs dans chaque partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Features : {X.shape[1]}  |  Train : {len(X_train)}  |  Test : {len(X_test)}')
print(f'Taux de risque -> Train : {y_train.mean()*100:.1f}%  |  Test : {y_test.mean()*100:.1f}%')

### A2. Entrainement des modeles

In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    r = {
        'modele': name,
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'rappel': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float('nan')
    }
    return r

results = []

# --- Baseline majoritaire ---
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
results.append(evaluate('Baseline majoritaire', y_test, baseline.predict(X_test)))

# --- Regression logistique ---
logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
])
logistic.fit(X_train, y_train)
logistic_prob = logistic.predict_proba(X_test)[:, 1]
results.append(evaluate('Logistic Regression', y_test, logistic.predict(X_test), logistic_prob))

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=400, class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
results.append(evaluate('Random Forest', y_test, rf.predict(X_test), rf_prob))

metrics_df = pd.DataFrame(results).set_index('modele')
print('=== Metriques comparees ===')
print(metrics_df.to_string())

os.makedirs('data/processed', exist_ok=True)
metrics_df.reset_index().to_csv('data/processed/tp3_model_metrics.csv', index=False)

In [ ]:
# Feature importance du Random Forest (Top 10)
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(fi['feature'][::-1], fi['importance'][::-1], color='#1A73E8', edgecolor='none')
ax.set_xlabel('Importance (Gini)')
ax.set_title('Top 10 des variables explicatives — Random Forest')
ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
ax.set_xlim(0, fi['importance'].max() * 1.25)
fig.tight_layout()
fig.savefig('report/assets/tp3_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

fi.to_csv('data/processed/tp3_feature_importance.csv', index=False)

---

## BRANCHE B — Deep Learning Vision (CNN TensorFlow)

### B1. Generation du dataset d'images synthetiques

120 images RGB 64×64 px, 2 classes : Cercle (0) et Rectangles (1). Cette brique demonstre la faisabilite technique d'une chaine vision dans le meme pipeline que la branche tabulaire.

In [ ]:
IMAGE_SIZE = 64

def generate_images(n=120, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    imgs = np.zeros((n, IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.float32)
    labels = np.array([i % 2 for i in range(n)], dtype=np.int32)
    for i in range(n):
        bg = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 0.14, dtype=np.float32)
        img = np.clip(bg + rng.normal(0, 0.02, bg.shape).astype(np.float32), 0, 1)
        if labels[i] == 0:
            cx, cy, r = rng.integers(24, 40), rng.integers(24, 40), rng.integers(12, 18)
            yy, xx = np.ogrid[:IMAGE_SIZE, :IMAGE_SIZE]
            img[(xx - cx)**2 + (yy - cy)**2 <= r**2] = [0.84, 0.23, 0.17]
        else:
            for _ in range(rng.integers(4, 9)):
                x, y = rng.integers(4, 50), rng.integers(4, 50)
                w, h = rng.integers(5, 12), rng.integers(5, 12)
                img[y:y+h, x:x+w] = rng.uniform(0.2, 0.95, 3).astype(np.float32)
        imgs[i] = img
    return imgs, labels

X_imgs, y_imgs = generate_images(120)
print(f'Dataset images : {X_imgs.shape}  |  Classes : {np.bincount(y_imgs)}')

In [ ]:
# Visualisation d'exemples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for col, (i0, i1) in enumerate(zip(np.where(y_imgs==0)[0][:5], np.where(y_imgs==1)[0][:5])):
    axes[0, col].imshow(X_imgs[i0]); axes[0, col].set_title('Cercle', fontsize=8); axes[0, col].axis('off')
    axes[1, col].imshow(X_imgs[i1]); axes[1, col].set_title('Rectangles', fontsize=8); axes[1, col].axis('off')
fig.suptitle('Exemples du dataset synthetique', fontsize=11, fontweight='bold')
fig.tight_layout()
os.makedirs('report/assets', exist_ok=True)
fig.savefig('report/assets/tp4_cnn_samples.png', dpi=150, bbox_inches='tight')
plt.show()

### B2. Architecture CNN et entrainement

In [ ]:
from sklearn.model_selection import train_test_split as tts
X_tr, X_val, y_tr, y_val = tts(X_imgs, y_imgs, test_size=0.2, random_state=RANDOM_STATE, stratify=y_imgs)
print(f'Train CNN : {len(X_tr)}  |  Val CNN : {len(X_val)}')

In [ ]:
# Architecture CNN sequentielle
cnn = models.Sequential([
    layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    layers.Conv2D(16, (3, 3), activation='relu'),   # Bloc 1 : detection de bords
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu'),   # Bloc 2 : detection de formes
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')           # Sortie binaire
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
history = cnn.fit(X_tr, y_tr, epochs=10, validation_data=(X_val, y_val), verbose=1)

history_df = pd.DataFrame(history.history)
history_df.insert(0, 'epoch', range(1, 11))
history_df.to_csv('data/processed/tp4_cnn_history.csv', index=False)

val_loss, val_acc = cnn.evaluate(X_val, y_val, verbose=0)
print(f'\nAccuracy validation : {val_acc:.4f}  |  Loss validation : {val_loss:.4f}')

In [ ]:
# Courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_df['epoch'], history_df[metric], color='#1A73E8', linewidth=2, label='Train')
    ax.plot(history_df['epoch'], history_df[f'val_{metric}'], color='#D93025',
            linewidth=2, linestyle='--', label='Validation')
    ax.set_xlabel('Epoque'); ax.set_ylabel(title); ax.set_title(f'{title} par epoque'); ax.legend()
fig.suptitle('Courbes d\'apprentissage CNN (10 epoques)', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('report/assets/tp4_cnn_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('Modeles et figures sauvegardes.')